In [ ]:
import requests
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import text, create_engine
import os
from datetime import datetime, UTC

BASE_URL = "https://api.openweathermap.org/data/2.5/weather"

In [ ]:
#Load API key from .env file
load_dotenv()

api_key = os.getenv("OPENWEATHER_API_KEY")

#Retrieves information from .env file to create engine
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
database = os.getenv("DB_NAME")
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")

In [ ]:
engine = create_engine(
    f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}"
)

In [ ]:
create_cities_table = """
CREATE TABLE IF NOT EXISTS cities(
    city_id SERIAL PRIMARY KEY,
    openweather_id INTEGER UNIQUE,
    city_name VARCHAR(50) NOT NULL,
    country CHAR(2) NOT NULL,
    latitude DECIMAL(8,5),
    longitude DECIMAL(8,5)
);
"""

insert_city_sql = """
INSERT INTO cities (
    openweather_id,
    city_name,
    country,
    latitude,
    longitude
)
VALUES (
    :openweather_id,
    :city_name,
    :country,
    :latitude,
    :longitude
)

RETURNING city_id
"""

get_city_id = """
SELECT city_id 
FROM cities
WHERE openweather_id = :openweather_id
"""


In [ ]:
create_observation_table = """
CREATE TABLE IF NOT EXISTS observations (
    observation_id SERIAL PRIMARY KEY,
    city_id INTEGER NOT NULL
        REFERENCES cities(city_id),
    observation_time TIMESTAMP NOT NULL,

    temperature DECIMAL(5,2),
    feels_like DECIMAL(5,2),
    temp_min DECIMAL(5,2),
    temp_max DECIMAL(5,2),

    humidity INTEGER,
    pressure INTEGER,

    wind_speed DECIMAL(5,2),
    wind_direction INTEGER,
    wind_gust DECIMAL(5,2),

    cloudiness INTEGER,

    weather_main VARCHAR(30),
    weather_description VARCHAR(100),
    visibility INTEGER
)
"""

insert_observation_sql = """
    INSERT INTO observations (
    city_id,
    observation_time,
    temperature,
    feels_like,
    temp_min,
    temp_max,
    humidity,
    pressure,
    wind_speed,
    wind_direction,
    wind_gust,
    cloudiness,
    weather_main,
    weather_description
    )
    VALUES (
    :city_id,
    :observation_time,
    :temperature,
    :feels_like,
    :temp_min,
    :temp_max,
    :humidity,
    :pressure,
    :wind_speed,
    :wind_direction,
    :wind_gust,
    :cloudiness,
    :weather_main,
    :weather_description 
    )

    RETURNING observation_id
"""

In [ ]:
#Get request to the API to receive the weather in specificed city

def get_weather(city: str) -> dict:
    params = {
    "q": city,
    "appid": api_key,
    "units": "metric"
    }

    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()
    return response.json()

#builds the parameter for get_or_create_city function
def build_city_param(weather: dict) -> dict:
    city_param = {
    "openweather_id": weather["id"],
    "city_name": weather["name"],
    "country": weather["sys"]["country"],
    "latitude": weather["coord"]["lat"],
    "longitude": weather["coord"]["lon"]
    }

    return city_param

#returns city_id and imports the city to cities table if it doesn't already exist
def get_or_create_city(engine, city_param: dict) -> int:
    with engine.begin() as conn:
        city_id = conn.execute(text(get_city_id),city_param).scalar()

        if city_id is None:
            city_id = conn.execute(
                text(insert_city_sql),
                city_param
            ).scalar()

    return city_id    


 #builds the parameter for observations
def build_observation_param(weather: dict, city_id:int) -> dict:
    observation_param = {
        "city_id": city_id,
        "temperature": weather["main"]["temp"],
        "feels_like": weather["main"]["feels_like"],
        "temp_min": weather["main"]["temp_min"],
        "temp_max": weather["main"]["temp_max"],
        "humidity": weather["main"]["humidity"],
        "pressure": weather["main"]["pressure"],
        "wind_speed": weather["wind"]["speed"],
        "wind_direction": weather["wind"]["deg"],
        "wind_gust": weather["wind"].get("gust"),
        "cloudiness": weather["clouds"]["all"],
        "weather_main": weather["weather"][0]["main"],
        "weather_description": weather["weather"][0]["description"],
        "observation_time": datetime.fromtimestamp(weather["dt"],UTC)
    }
    
    return observation_param

#creates observation in primary table
def insert_observation(engine, observation_param: dict) -> int:
    with engine.begin() as conn:
        observation_id = conn.execute(
            text(insert_observation_sql),
            observation_param
        ).scalar()
        
    
    return observation_id       